# 2. Train Static Sign Model (CNN-1D)

**Kiến trúc**: 3× Conv1D blocks + GlobalAvgPool + Dense

**Input**: Hand landmarks 126 features (21 landmarks × 3 coords × 2 hands)

**Tham khảo**:
- Paper ĐH Surabaya (Jurnal RESTI): CNN-1D **96.93%** vs MLP 91.2% trên hand keypoints
- Paper ĐH HUST 2025 (JST): MediaPipe + CNN > 95% cho VSL alphabet

> Chạy notebook `01_data_preparation.ipynb` trước để có data.

In [ ]:
# === CẤU HÌNH ===
DATA_DIR = "../data/processed/static"
OUTPUT_DIR = "../models"
EPOCHS = 30
BATCH_SIZE = 64
AUGMENT_FACTOR = 3   # Số lần augment (tăng dataset lên 4x)
VAL_SPLIT = 0.15
TEST_SPLIT = 0.10

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

## 2.1 Load và tiền xử lý data

In [ ]:
# --- Preprocessing functions ---
def normalize_landmarks(landmarks):
    """Chuẩn hóa landmarks: wrist làm gốc tọa độ, scale theo khoảng cách lớn nhất."""
    def _normalize_single(frame):
        result = frame.copy()
        for hand_offset in [0, 63]:
            hand = result[hand_offset:hand_offset + 63]
            if np.all(hand == 0):
                continue
            points = hand.reshape(21, 3)
            wrist = points[0].copy()
            points -= wrist
            max_dist = np.max(np.linalg.norm(points, axis=1))
            if max_dist > 0:
                points /= max_dist
            result[hand_offset:hand_offset + 63] = points.flatten()
        return result

    if landmarks.ndim == 1:
        return _normalize_single(landmarks)
    return np.array([_normalize_single(f) for f in landmarks])


def augment_landmarks(landmarks, seed=None):
    """Augment: random scale (±10%), rotation (±15°), noise. KHÔNG flip ngang."""
    rng = np.random.RandomState(seed)
    result = landmarks.copy()

    # Scale
    result *= rng.uniform(0.9, 1.1)

    # Rotation (x-y plane)
    angle = rng.uniform(-15, 15) * np.pi / 180
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    for hand_offset in [0, 63]:
        hand = result[hand_offset:hand_offset + 63]
        if np.all(hand == 0):
            continue
        points = hand.reshape(21, 3)
        x_new = points[:, 0] * cos_a - points[:, 1] * sin_a
        y_new = points[:, 0] * sin_a + points[:, 1] * cos_a
        points[:, 0], points[:, 1] = x_new, y_new
        result[hand_offset:hand_offset + 63] = points.flatten()

    # Noise
    result += rng.normal(0, 0.002, result.shape)
    return result


# --- Load data ---
X, y, labels = [], [], {}
class_dirs = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])

for idx, class_name in enumerate(class_dirs):
    labels[idx] = class_name
    class_path = os.path.join(DATA_DIR, class_name)
    for fname in os.listdir(class_path):
        if fname.endswith(".npy"):
            data = np.load(os.path.join(class_path, fname))
            if data.ndim == 1 and len(data) == 126:
                X.append(data)
                y.append(idx)

X, y = np.array(X), np.array(y)
print(f"Loaded: {len(X)} samples, {len(labels)} classes")

# Normalize
X = normalize_landmarks(X)

# Train/Val/Test split
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SPLIT, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=VAL_SPLIT / (1 - TEST_SPLIT),
    random_state=42, stratify=y_trainval
)

# Augment training data
X_aug, y_aug = [X_train], [y_train]
for i in range(AUGMENT_FACTOR):
    augmented = np.array([augment_landmarks(x, seed=i * len(X_train) + j)
                          for j, x in enumerate(X_train)])
    X_aug.append(augmented)
    y_aug.append(y_train)
X_train, y_train = np.concatenate(X_aug), np.concatenate(y_aug)

# Reshape for Conv1D: (N, 126) → (N, 126, 1)
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

## 2.2 Build model CNN-1D

In [ ]:
def build_static_model(num_classes, input_dim=126):
    """CNN-1D: 3 conv blocks + GlobalAvgPool + Dense head."""
    inputs = keras.Input(shape=(input_dim, 1), name="landmarks")

    x = layers.Conv1D(64, 3, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv1D(128, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(256, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(num_classes, activation="softmax", name="output")(x)

    return keras.Model(inputs=inputs, outputs=x, name="static_cnn1d")


num_classes = len(labels)
model = build_static_model(num_classes)
model.summary()

## 2.3 Training

In [ ]:
# Class weights (xử lý imbalanced data - từ AIO2025)
class_weights_arr = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = {i: w for i, w in enumerate(class_weights_arr)}

# Compile với AdamW (từ AIO2025)
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.001, weight_decay=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
]

# Train!
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

## 2.4 Đánh giá kết quả

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["accuracy"], label="Train")
ax1.plot(history.history["val_accuracy"], label="Validation")
ax1.set_title("Accuracy")
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(True)

ax2.plot(history.history["loss"], label="Train")
ax2.plot(history.history["val_loss"], label="Validation")
ax2.set_title("Loss")
ax2.set_xlabel("Epoch")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Test evaluation
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
# Confusion matrix + Classification report
y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
label_names = [labels[i].split("_", 1)[1] if "_" in labels[i] else labels[i] for i in range(num_classes)]

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_names))

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=label_names)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix - Static CNN-1D")
plt.tight_layout()
plt.show()

## 2.5 Lưu model

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save model
model_path = os.path.join(OUTPUT_DIR, "static_cnn1d.keras")
model.save(model_path)
print(f"Model saved: {model_path}")

# Save labels
labels_path = os.path.join(OUTPUT_DIR, "static_labels.json")
with open(labels_path, "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in labels.items()}, f, ensure_ascii=False, indent=2)
print(f"Labels saved: {labels_path}")

# Save training history
hist_path = os.path.join(OUTPUT_DIR, "static_history.json")
hist_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with open(hist_path, "w") as f:
    json.dump(hist_data, f)
print(f"History saved: {hist_path}")

print(f"\nFinal test accuracy: {test_acc:.4f}")

---
**Tiếp theo**: Chạy notebook `03_train_dynamic_model.ipynb` để train Bi-LSTM + Attention cho dynamic signs.